# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

The dataset details protein abundance, modifications, and sequences from mass spectrometry analysis of extracellular vesicles from stimulated human mast cells.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and initialize access to the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Print out high-level metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")
print(f"Version: {meta.version}")
print(f"Cite As: {getattr(meta, 'citeAs', getattr(meta, 'citation', None))}")

## 2. Data Overview
Inspect the dataset structure and discover the available record sets and their fields (all referenced by their `@id`).

In [ ]:
print("Available record sets (by @id):\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")
    print()

### Example Record Printing
Below, we print 1-2 example records from the first available record set, using its `@id`. You can reference the chosen record set's `@id` in later sections.

In [ ]:
# You may need to select the record set @id found above
# We'll extract the record set ids into a list for convenience

record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record sets found in dataset: {record_set_ids}\n")

# Print out 2 sample records from the first record set
example_record_set = record_set_ids[0]
for i, record in enumerate(dataset.records(record_set=example_record_set)):
    print(record)
    if i >= 1:
        break

## 3. Data Extraction
Extract all data from each record set into a pandas DataFrame, using record set and field `@id`s from above.

In [ ]:
# Extract all dataframes from available record sets
dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    dataframes[rec_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set @id: {rec_id}")
    if len(records) > 0:
        print(f"Fields/Columns: {dataframes[rec_id].columns.tolist()}")
    print()

# Pick one record set to use for the rest of the notebook
main_record_set = record_set_ids[0]
main_df = dataframes[main_record_set]
print(f"Using main record set for further analysis: {main_record_set}")

Inspect the head and columns of the main record set.

In [ ]:
print(f"Columns in main record set ({main_record_set}):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's do some simple data processing. We'll select a numeric field for analysis, filter and normalize, all while referencing fields by their `@id` as available.

In [ ]:
# For demonstration, pick a numeric field used for filtering/EDA
# Let's try to auto-detect likely numeric fields
print("Searching for a numeric column (float or integer)...")
numeric_candidates = []
for field in main_df.columns:
    try:
        # Try converting to numeric with coercion
        col = pd.to_numeric(main_df[field], errors='coerce')
        if col.notna().sum() > 0:
            numeric_candidates.append(field)
    except Exception:
        pass
print(f"Numeric candidates: {numeric_candidates}")

# Pick the 'coverage' or first candidate for the demo
if 'coverage' in numeric_candidates:
    numeric_field = 'coverage'
else:
    numeric_field = numeric_candidates[0] if numeric_candidates else main_df.columns[0]

# Convert the column to float if possible
main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')

print(f"Using field for numeric analysis: '{numeric_field}'\n")

# Filtering
threshold = main_df[numeric_field].quantile(0.50)  # median, as an example
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3f} (median): {len(filtered_df)} records\n")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another field (try for 'sample' or first non-numeric field)
non_numeric_candidates = [c for c in main_df.columns if c != numeric_field and main_df[c].dtype == 'object']
group_field = None
if 'sample' in main_df.columns:
    group_field = 'sample'
elif len(non_numeric_candidates) > 0:
    group_field = non_numeric_candidates[0]

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
    print(f"\nGrouped data by field '{group_field}' (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if present, average values grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by group_field (if available)
if group_field is not None:
    plt.figure(figsize=(10,4))
    sns.boxplot(
        data=filtered_df,
        x=group_field, y=numeric_field
    )
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the FAIR<sup>2</sup> dataset of human protein mass spectrometry analysis. Referencing record sets and fields by their Croissant `@id`, we loaded the schema, extracted tabular data, performed basic filtering and normalization, and visualized features. This foundation enables deeper domain analysis, such as identifying protein abundance patterns, examining coverage distributions, or preparing the data for advanced proteomics studies.

For further work, see the [Croissant specification](https://mlcommons.github.io/croissant/) and explore the dataset's original documentation.